# ⚡ Edge-AI ve Düşük Gecikmeli XAI Optimizasyonu
### *TreeSHAP, FastLinearSHAP ve Örnekleme Tabanlı Hızlandırma Karşılaştırmaları*

Oyun teorisi tabanlı açıklanabilirlik algoritmalarının en büyük engeli olan $O(2^N)$ hesaplama patlamasının nasıl aşıldığını ve gömülü/uç birimlerde (Edge-AI) milisaniyelik gecikmelerin nasıl sağlandığını inceler.

In [ ]:
import time
import numpy as np
from sklearn.linear_model import Ridge
import xgboost as xgb
from explainers.cost_optimizer import FastLinearSHAP, FastTreeSHAP, CoalitionSampler, benchmark_shap_latency

print('Optimizasyon ve Benchmark Modülleri Yüklendi.')

## 1. FastLinearSHAP Kapalı Form Hızlandırması: O(2^d) yerine O(d)

In [ ]:
d = 20
N = 5000
X = np.random.randn(N, d)
y = X @ np.random.uniform(0.5, 2.0, d) + np.random.randn(N) * 0.1

ridge = Ridge().fit(X, y)

t0 = time.perf_counter()
linear_explainer = FastLinearSHAP.from_sklearn(ridge, X)
res = linear_explainer.explain(X)
t_elapsed = (time.perf_counter() - t0) * 1000.0

print(f'FastLinearSHAP: {N} örnek için toplam süre: {t_elapsed:.2f} ms')
print(f'Örnek başına gecikme: {t_elapsed / N * 1000.0:.2f} mikrosaniye')
print(f'Hesaplanan Shapley Değer Boyutu: {res["values"].shape}')

## 2. TreeSHAP Gecikme Benchmarkı

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=50, max_depth=4).fit(X[:1000], y[:1000])
bench = benchmark_shap_latency(xgb_model, X, n_instances=100)

print(f"Toplam Süre: {bench['total_time_ms']:.2f} ms")
print(f"Örnek Başına Gecikme: {bench['ms_per_sample']:.2f} ms")
print(f"Edge-AI Gerçek Zamanlı Uyumluluk (<5ms): {'EVET ✅' if bench['edge_ready'] else 'HAYIR ❌'}")